# MiniMax H3 (ComfyUI) I2V Colab 検証用

新 I2V ワークフロー（`video_minimax_h3_i2v.json`）専用・外部入力（プロンプト／画像）反映対応のノートブックです。古い T2V ワークフローは使用しません。

### 外部入力の入れ方（優先順）
1. セル 3 の `PROMPT_OVERRIDE` / `IMAGE_OVERRIDE`
2. Google Drive `ComfyUI/input/prompt.txt`（テキストのみ上書き）
3. Google Drive `ComfyUI/input/i2v_input.json`（`batches/minimax_h3_i2v_input.json` と同じ JSON 形式）
4. リポジトリ既定の `batches/minimax_h3_i2v_input.json`

### 手順
1. **「ランタイム ＞ ランタイムのタイプを変更」** で **A100 GPU** を選択して接続します。
2. セル 1〜4 を上から順に実行し、GUI で `workflow_api.json` を `Export (API)` すると CLI バッチ実行にもそのまま使えます。
3. 終了時は必ず **セル 6 (`drive.flush_and_unmount()`)** を実行し、ランタイムを削除して課金を停止してください。


In [ ]:
import shutil
import subprocess

if not shutil.which("nvidia-smi"):
    raise RuntimeError("GPU が割り当てられていません (CPU ランタイム)。ランタイムのタイプを A100 へ変更してください。")

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
).strip()
print("GPU:", gpu_info)
if "A100" not in gpu_info and "G4" not in gpu_info and "H100" not in gpu_info:
    print("[WARN] A100/G4/H100 以外のGPUです。MiniMax H3 (~42GB) は低速またはメモリ不足の可能性があります。")

print(subprocess.check_output(["free", "-h"], text=True).strip())
print(subprocess.check_output(["df", "-h", "/"], text=True).strip())

In [ ]:
from pathlib import Path

from google.colab import drive

DRIVE_ROOT = "ComfyUI"
REQUIRED_ASSETS = [
    "models/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
    "models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    "models/vae/minimax_h3_video_vae_int8_convrot.safetensors",
    "models/vae/minimax_h3_audio_vae_fp32.safetensors",
    "models/loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors",
]

drive.mount("/content/drive")
drive_base = Path("/content/drive/MyDrive") / DRIVE_ROOT
if not drive_base.is_dir():
    raise FileNotFoundError(f"Drive root が見つかりません: {drive_base}")

missing = []
for rel in REQUIRED_ASSETS:
    target = drive_base / rel
    if not target.is_file():
        missing.append(str(target))
    else:
        size_gb = target.stat().st_size / (1024**3)
        print(f"[OK] {rel} ({size_gb:.2f} GiB)")

if missing:
    raise FileNotFoundError("不足しているモデル資産があります:\n" + "\n".join(missing))
print("全必須モデル 5 ファイルの確認が完了しました。")

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import urllib.request

# 外部入力の上書き設定（空なら Drive の input/i2v_input.json -> input/prompt.txt -> リポ batches/ を自動読込）
PROMPT_OVERRIDE = ""
IMAGE_OVERRIDE = ""  # 例: "drive_assets/my_image.png"
DURATION_OVERRIDE = None  # 例: 5
SEED_OVERRIDE = None  # 例: 12345
CIVITAI_TOKEN = ""  # Colab シークレット "CIVITAI_TOKEN" があれば自動読込
DOWNLOAD_LORAS = {
    3449495: "H3_VBVR_Pro-V2.safetensors",
    3405852: "H3_DR34ML4Y_P1.safetensors",
}

INSTALL_SAGEATTENTION = False
DRIVE_ROOT = "ComfyUI"
COMFY_DIR = Path("/content/ComfyUI")
DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_ROOT

RAW_BASES = [
    "https://raw.githubusercontent.com/xzyozi/colab-cli-comfyui-runner/feat/minimax-h3-phase1",
    "https://raw.githubusercontent.com/xzyozi/colab-cli-comfyui-runner/develop",
]
FALLBACK_NODES = [
    "https://github.com/kijai/ComfyUI-KJNodes.git",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "https://github.com/rgthree/rgthree-comfy.git",
]


def fetch_first(urls: list[str]) -> tuple[str, bytes] | tuple[None, None]:
    for url in urls:
        try:
            with urllib.request.urlopen(url, timeout=30) as resp:
                return url, resp.read()
        except Exception as exc:
            print(f"Fetch failed ({url}): {exc}")
    return None, None


def load_external_input(drive_base: Path) -> dict:
    ext = {"prompt": None, "image": None, "duration_sec": None, "seed": None}
    drive_json = drive_base / "input" / "i2v_input.json"
    drive_txt = drive_base / "input" / "prompt.txt"
    payload = None
    source_label = "workflow default"

    if drive_json.is_file():
        payload = json.loads(drive_json.read_text(encoding="utf-8"))
        source_label = str(drive_json)
    else:
        url, raw = fetch_first([f"{b}/batches/minimax_h3_i2v_input.json" for b in RAW_BASES])
        if raw is not None:
            payload = json.loads(raw.decode("utf-8"))
            source_label = url or "batches/minimax_h3_i2v_input.json"

    if isinstance(payload, dict):
        job = payload["jobs"][0] if isinstance(payload.get("jobs"), list) and payload["jobs"] else payload
        if isinstance(job.get("prompt"), str) and job["prompt"].strip():
            ext["prompt"] = job["prompt"]
        if isinstance(job.get("duration_sec"), (int, float)):
            ext["duration_sec"] = float(job["duration_sec"])
        if isinstance(job.get("seed"), int):
            ext["seed"] = job["seed"]
        assets = job.get("input_assets") or {}
        first = assets.get("first_frame") if isinstance(assets, dict) else None
        if isinstance(first, dict) and first.get("path"):
            ext["image"] = f"drive_assets/{first['path']}" if first.get("source") == "drive" else str(first["path"])
        elif isinstance(job.get("image"), str) and job["image"].strip():
            ext["image"] = job["image"].strip()

    if drive_txt.is_file():
        txt = drive_txt.read_text(encoding="utf-8").strip()
        if txt:
            ext["prompt"] = txt
            source_label = f"{source_label} + {drive_txt}"

    if PROMPT_OVERRIDE.strip():
        ext["prompt"] = PROMPT_OVERRIDE.strip()
        source_label = "PROMPT_OVERRIDE"
    if IMAGE_OVERRIDE.strip():
        ext["image"] = IMAGE_OVERRIDE.strip()
    if DURATION_OVERRIDE is not None:
        ext["duration_sec"] = float(DURATION_OVERRIDE)
    if SEED_OVERRIDE is not None:
        ext["seed"] = int(SEED_OVERRIDE)

    print("External input source:", source_label)
    return ext


def inject_i2v_inputs(wf_bytes: bytes, ext: dict) -> bytes:
    wf = json.loads(wf_bytes.decode("utf-8"))
    nodes = {n["id"]: n for n in wf.get("nodes", [])}
    if ext.get("prompt"):
        if 173 in nodes:
            nodes[173]["widgets_values"] = [ext["prompt"]]
        if 130 in nodes and isinstance(nodes[130].get("widgets_values"), list):
            w = list(nodes[130]["widgets_values"])
            w[0] = ext["prompt"]
            nodes[130]["widgets_values"] = w
        print(f"Injected prompt ({len(ext['prompt'])} chars)")
    if ext.get("image") and 182 in nodes:
        nodes[182]["widgets_values"] = [ext["image"], "image"]
        print("Injected first_frame image:", ext["image"])
    if ext.get("duration_sec") is not None and 132 in nodes:
        nodes[132]["widgets_values"] = [ext["duration_sec"]]
    if ext.get("seed") is not None and 154 in nodes:
        nodes[154]["widgets_values"] = [ext["seed"], "fixed"]
    return (json.dumps(wf, indent=2, ensure_ascii=False) + "\n").encode("utf-8")


AUTOLOAD_INIT = """WEB_DIRECTORY = "./js"
NODE_CLASS_MAPPINGS = {}
NODE_DISPLAY_NAME_MAPPINGS = {}
__all__ = ["NODE_CLASS_MAPPINGS", "NODE_DISPLAY_NAME_MAPPINGS", "WEB_DIRECTORY"]
"""

if not shutil.which("uv"):
    subprocess.run(["pip", "install", "-q", "uv"], check=True)
if not shutil.which("comfy"):
    subprocess.run(["uv", "pip", "install", "--system", "-q", "comfy-cli"], check=True)

if not COMFY_DIR.is_dir():
    res = subprocess.run(
        ["comfy", "--skip-prompt", "--workspace", str(COMFY_DIR), "install", "--nvidia", "--fast-deps"],
        check=False,
    )
    if res.returncode != 0:
        shutil.rmtree(COMFY_DIR, ignore_errors=True)
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY_DIR)],
            check=True,
        )
        subprocess.run(
            ["uv", "pip", "install", "--system", "-q", "-r", str(COMFY_DIR / "requirements.txt")],
            check=True,
        )

if INSTALL_SAGEATTENTION:
    subprocess.run(["uv", "pip", "install", "--system", "-q", "sageattention"], check=True)

nodes_url, nodes_bytes = fetch_first([f"{b}/workflows/minimax_h3/nodes.txt" for b in RAW_BASES])
if nodes_bytes is not None:
    node_repos = [
        line.strip()
        for line in nodes_bytes.decode("utf-8").splitlines()
        if line.strip() and not line.strip().startswith("#")
    ]
    print(f"Loaded nodes.txt from: {nodes_url}")
else:
    node_repos = FALLBACK_NODES

custom_nodes_dir = COMFY_DIR / "custom_nodes"
custom_nodes_dir.mkdir(parents=True, exist_ok=True)
for repo_url in node_repos:
    repo_name = repo_url.rstrip("/").rsplit("/", 1)[-1]
    if repo_name.endswith(".git"):
        repo_name = repo_name[:-4]
    target_dir = custom_nodes_dir / repo_name
    if not target_dir.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(target_dir)], check=True)
    req_file = target_dir / "requirements.txt"
    if req_file.is_file():
        subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", str(req_file)], check=True)

for subdir in ("diffusion_models", "text_encoders", "vae", "loras", "embeddings"):
    src = DRIVE_BASE / "models" / subdir
    dst = COMFY_DIR / "models" / subdir
    if src.is_dir():
        if dst.is_symlink() or dst.exists():
            shutil.rmtree(dst) if dst.is_dir() and not dst.is_symlink() else dst.unlink()
        os.symlink(src, dst)

(DRIVE_BASE / "output").mkdir(parents=True, exist_ok=True)
token = CIVITAI_TOKEN.strip()
if not token:
    try:
        from google.colab import userdata

        token = (userdata.get("CIVITAI_TOKEN") or "").strip()
    except Exception:
        token = ""

if token:
    lora_dir = DRIVE_BASE / "models" / "loras"
    lora_dir.mkdir(parents=True, exist_ok=True)
    for vid, fname in DOWNLOAD_LORAS.items():
        out_path = lora_dir / fname
        if out_path.is_file():
            print(f"[SKIP] LoRA already exists: {fname}")
            continue
        dl_url = f"https://civitai.red/api/download/models/{vid}?token={token}"
        print(f"Downloading LoRA: {fname} ...")
        subprocess.run(["curl", "-L", "--fail", "-o", str(out_path), dl_url], check=True)
out_dst = COMFY_DIR / "output"
if out_dst.is_symlink() or out_dst.exists():
    shutil.rmtree(out_dst) if out_dst.is_dir() and not out_dst.is_symlink() else out_dst.unlink()
os.symlink(DRIVE_BASE / "output", out_dst)

if (DRIVE_BASE / "input").is_dir():
    (COMFY_DIR / "input").mkdir(parents=True, exist_ok=True)
    in_dst = COMFY_DIR / "input" / "drive_assets"
    if in_dst.is_symlink() or in_dst.exists():
        shutil.rmtree(in_dst) if in_dst.is_dir() and not in_dst.is_symlink() else in_dst.unlink()
    os.symlink(DRIVE_BASE / "input", in_dst)

user_wf_dir = COMFY_DIR / "user" / "default" / "workflows"
user_wf_dir.mkdir(parents=True, exist_ok=True)
ext_dir = COMFY_DIR / "custom_nodes" / "ccr_autoload"
js_dir = ext_dir / "js"
js_dir.mkdir(parents=True, exist_ok=True)

# 古い T2V ワークフローが残っていれば確実に排除
for old_path in (user_wf_dir / "video_minimax_h3_t2v.json", js_dir / "video_minimax_h3_t2v.json"):
    if old_path.exists():
        old_path.unlink()

used_url, wf_bytes = fetch_first([f"{b}/workflows/minimax_h3/video_minimax_h3_i2v.json" for b in RAW_BASES])
if wf_bytes is None:
    raise RuntimeError("video_minimax_h3_i2v.json の取得に失敗しました。")
print(f"Downloaded video_minimax_h3_i2v.json from: {used_url}")

ext_input = load_external_input(DRIVE_BASE)
injected_bytes = inject_i2v_inputs(wf_bytes, ext_input)
wf_sig = hashlib.sha256(injected_bytes).hexdigest()[:12]

(user_wf_dir / "video_minimax_h3_i2v.json").write_bytes(injected_bytes)
(js_dir / "video_minimax_h3_i2v.json").write_bytes(injected_bytes)

autoload_js = f"""import {{ app }} from "../../scripts/app.js";

app.registerExtension({{
  name: "ccr.autoload.minimax_h3",
  async setup() {{
    setTimeout(async () => {{
      try {{
        const key = "ccr_autoloaded_i2v_{wf_sig}";
        if (sessionStorage.getItem(key) === "1") {{
          return;
        }}
        const wfUrl = new URL("./video_minimax_h3_i2v.json", import.meta.url);
        const resp = await fetch(wfUrl, {{ cache: "no-store" }});
        if (!resp.ok) {{
          return;
        }}
        const wf = await resp.json();
        await app.loadGraphData(wf);
        sessionStorage.setItem(key, "1");
      }} catch (err) {{
        console.warn("[ccr_autoload] error:", err);
      }}
    }}, 600);
  }},
}});
"""

settings_path = COMFY_DIR / "user" / "default" / "comfy.settings.json"
settings_path.write_text(json.dumps({"Comfy.DevMode": True}, indent=2) + "\n", encoding="utf-8")
(ext_dir / "__init__.py").write_text(AUTOLOAD_INIT, encoding="utf-8")
(js_dir / "autoload.js").write_text(autoload_js, encoding="utf-8")
print(f"I2V ワークフロー・外部入力反映・環境設定が完了しました (sig={wf_sig})。")

In [ ]:
from pathlib import Path
import subprocess
import sys
import time
from urllib import request as urllib_request

from google.colab import output

COMFY_PORT = 8188
RESTART_IF_RUNNING = True
LOG_PATH = Path("/tmp/comfyui.log")

pgrep = subprocess.run(["pgrep", "-f", f"[m]ain.py.*--port {COMFY_PORT}"], capture_output=True, text=True)
if pgrep.returncode == 0 and RESTART_IF_RUNNING:
    for pid in pgrep.stdout.split():
        subprocess.run(["kill", pid], check=False)
    time.sleep(2)
    pgrep = subprocess.run(["pgrep", "-f", f"[m]ain.py.*--port {COMFY_PORT}"], capture_output=True, text=True)

if pgrep.returncode != 0:
    with LOG_PATH.open("wb") as log_file:
        subprocess.Popen(
            [
                sys.executable,
                "/content/ComfyUI/main.py",
                "--listen",
                "127.0.0.1",
                "--port",
                str(COMFY_PORT),
                "--enable-cors-header",
            ],
            stdin=subprocess.DEVNULL,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

ready = False
for _ in range(36):
    try:
        with urllib_request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=3) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(5)

if not ready:
    print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:])
    raise RuntimeError("ComfyUI が 180 秒以内に起動しませんでした。")

print("ComfyUI 起動完了。以下のリンクから GUI を開いてください（I2V 専用 & 外部入力反映済み）：")
proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({COMFY_PORT})")
print("Proxy URL:", proxy_url)
output.serve_kernel_port_as_window(COMFY_PORT)

In [ ]:
from pathlib import Path

DRIVE_ROOT = "ComfyUI"
out_dir = Path("/content/drive/MyDrive") / DRIVE_ROOT / "output"
print("=== Output files ===")
for item in sorted(out_dir.rglob("*")):
    if item.is_file():
        print(f"{item.relative_to(out_dir)} ({item.stat().st_size / (1024**2):.2f} MiB)")

log_path = Path("/tmp/comfyui.log")
if log_path.is_file():
    print("\n=== ComfyUI log (last 40 lines) ===")
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(lines[-40:]))

In [ ]:
from google.colab import drive

drive.flush_and_unmount()
print("DriveFS フラッシュ・アンマウント完了。Colab メニューからランタイムを削除して課金を停止してください。")